<a href="https://colab.research.google.com/github/xyaoaf/UrbanTreeStudywithLidar/blob/main/230304TreePopulationData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
import numpy as np
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

In [ ]:
# load your CSV file into a pandas DataFrame
dfEast = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/East009701_TreeMetrics.csv')
dfEastArc = pd.read_csv('//content/drive/MyDrive/XihanYAO_GISDataShare/East009701_TreeMetrics_230305ArcPara.csv')
dfWest = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/West002401_TreeMetrics.csv')
dfWestArc = pd.read_csv('//content/drive/MyDrive/XihanYAO_GISDataShare/West002401_TreeMetrics_230305ArcPara.csv')

# specify the column(s) that define a unique feature
feature_cols = ['name']

# drop duplicates based on the feature columns and keep the first occurrence
dfEast.drop_duplicates(subset=feature_cols, keep='first', inplace=True)
dfWest.drop_duplicates(subset=feature_cols, keep='first', inplace=True)

In [ ]:
dfEast.tail(5)

In [ ]:
dfEastArc.tail(5)

In [ ]:
ids1 = set(dfEast['name'])
ids2 = set(dfEastArc['treeID'])

# Check if the sets are equal
if ids1 == ids2:
    print('The IDs match perfectly')
else:
    # Find the IDs that are in one set but not the other
    missing_ids = ids1.symmetric_difference(ids2)
    print(f'The following IDs are not present in both dataframes: {missing_ids}')

In [ ]:
# Drop rows I
dfEastArc = dfEastArc.drop(dfEastArc[dfEastArc['treeID'] == 2828].index)
dfEastArc = dfEastArc.drop(dfEastArc[dfEastArc['treeID'] == 4760].index)
dfEastArc = dfEastArc.drop(dfEastArc[dfEastArc['treeID'] == 5752].index)

# Print the updated DataFrame
# print(dfEastArc)

In [ ]:
ids1 = set(dfEast['name'])
ids2 = set(dfEastArc['treeID'])

# Check if the sets are equal
if ids1 == ids2:
    print('The IDs match perfectly')
else:
    # Find the IDs that are in one set but not the other
    missing_ids = ids1.symmetric_difference(ids2)
    print(f'The following IDs are not present in both dataframes: {missing_ids}')

In [ ]:
dfWest.tail(5)

In [ ]:
dfWestArc.tail(5)

In [ ]:
ids1 = set(dfWest['name'])
ids2 = set(dfWestArc['treeID'])

# Check if the sets are equal
if ids1 == ids2:
    print('The IDs match perfectly')
else:
    # Find the IDs that are in one set but not the other
    missing_ids = ids1.symmetric_difference(ids2)
    print(f'The following IDs are not present in both dataframes: {missing_ids}')

In [ ]:
dfEastMerge = pd.merge(dfEast, dfEastArc, left_on='name', right_on='treeID')
dfEastMerge = dfEastMerge.drop(dfEastMerge[dfEastMerge['treeID'] == 7638].index)
dfWestMerge = pd.merge(dfWest, dfWestArc, left_on='name', right_on='treeID')
dfEastMerge

In [ ]:
dfWestMerge

In [ ]:
print(len(dfEast))
print(len(dfEastArc))
print(len(dfEastMerge))
print(len(dfWest))
print(len(dfWestArc))
print(len(dfWestMerge))

In [ ]:
# check which tree names are missing
# create a range of expected index values
expected_idx = pd.RangeIndex(start=1, stop=7629)
# check which index values are missing
missing_idx = expected_idx.difference(dfEastMerge.name)
# display the missing index values
print('Missing index values:', missing_idx)

# create a range of expected index values
expected_idx = pd.RangeIndex(start=1, stop=5555)
# check which index values are missing
missing_idx = expected_idx.difference(dfWestMerge.name)
# display the missing index values
print('Missing index values:', missing_idx)

In [ ]:
dfWestMerge = dfWestMerge.drop(columns='Ground')
dfWestMerge = dfWestMerge.drop(columns='treeID')
dfWestMerge = dfWestMerge.drop(columns='OID_')
dfWestMerge = dfWestMerge.drop(columns='index')
dfWestMerge = dfWestMerge.drop(columns='winRadius')
dfEastMerge = dfEastMerge.drop(columns='treeID')
dfEastMerge = dfEastMerge.drop(columns='p95')
dfEastMerge = dfEastMerge.drop(columns='Ground')
dfEastMerge = dfEastMerge.drop(columns='OID_')
dfEastMerge = dfEastMerge.drop(columns='index')
dfEastMerge = dfEastMerge.drop(columns='winRadius')
dfEastMerge = dfEastMerge.rename(columns={'name': 'treeID'})
dfWestMerge = dfWestMerge.rename(columns={'name': 'treeID'})
dfWestMerge['height'] = dfWestMerge['height']*0.3048
dfEastMerge['height'] = dfEastMerge['height']*0.3048

In [ ]:
# merge two datasets together
dfWestMerge['treeID'] = dfWestMerge['treeID'] + 10000
dfEastMerge['treeID'] = dfEastMerge['treeID'] + 20000

In [ ]:
# check column names
print(dfEastMerge.columns)
print(dfWestMerge.columns)

## Add spectural bands

In [ ]:
import os

# define the folder path containing the CSV files
folder_path = "/content/drive/MyDrive/XihanYAO_GISDataShare/230308ZonalSpectrual"

# create an empty dictionary to store the DataFrames
dfs = {}
df_names=[]

# loop over each file in the folder
for file_name in os.listdir(folder_path):
    # check if the file is a CSV file
    if file_name.endswith(".csv"):
        # read the CSV file into a DataFrame
        df = pd.read_csv(os.path.join(folder_path, file_name))
        # name the DataFrame after the file name
        df_name = file_name.split(".")[0]
        df.name = df_name
        df_names.append(df.name)
        # add the DataFrame to the dictionary
        dfs[df_name] = df

# example: print the first few rows of the DataFrame with file name "example.csv"
print(dfs["e_2018_1_zonal"].head())

In [ ]:
dfs["e_2018_1_zonal"]
# sort the dictionary by key and create a new dictionary
dfs = dict(sorted(dfs.items()))
dfs.keys()

In [ ]:
df_names.sort()
df_names

In [ ]:
for key, df in list(dfs.items())[:16]:
  df['TREEID'] = df['TREEID'] + 10000
for key, df in list(dfs.items())[:8]:
  df['TREEID'] = df['TREEID'] + 10000

In [ ]:
dfs["e_2018_3_zonal"]

In [ ]:

df_2018_1_zonal = pd.concat([dfs["e_2018_1_zonal"], dfs["w_2018_1_zonal"]]).rename(columns={'MEAN': '2018_R'}).reset_index(drop=True).rename(columns={'TREEID': 'treeID'})
df_2018_2_zonal = pd.concat([dfs["e_2018_2_zonal"], dfs["w_2018_2_zonal"]]).rename(columns={'MEAN': '2018_G'}).reset_index(drop=True).rename(columns={'TREEID': 'treeID'})
df_2018_3_zonal = pd.concat([dfs["e_2018_3_zonal"], dfs["w_2018_3_zonal"]]).rename(columns={'MEAN': '2018_B'}).reset_index(drop=True).rename(columns={'TREEID': 'treeID'})
df_2018_4_zonal = pd.concat([dfs["e_2018_4_zonal"], dfs["w_2018_4_zonal"]]).rename(columns={'MEAN': '2018_N'}).reset_index(drop=True).rename(columns={'TREEID': 'treeID'})
df_2020_1_zonal = pd.concat([dfs["e_2020_1_zonal"], dfs["w_2020_1_zonal"]]).rename(columns={'MEAN': '2020_R'}).reset_index(drop=True).rename(columns={'TREEID': 'treeID'})
df_2020_2_zonal = pd.concat([dfs["e_2020_2_zonal"], dfs["w_2020_2_zonal"]]).rename(columns={'MEAN': '2020_G'}).reset_index(drop=True).rename(columns={'TREEID': 'treeID'})
df_2020_3_zonal = pd.concat([dfs["e_2020_3_zonal"], dfs["w_2020_3_zonal"]]).rename(columns={'MEAN': '2020_B'}).reset_index(drop=True).rename(columns={'TREEID': 'treeID'})
df_2020_4_zonal = pd.concat([dfs["e_2020_4_zonal"], dfs["w_2020_4_zonal"]]).rename(columns={'MEAN': '2020_N'}).reset_index(drop=True).rename(columns={'TREEID': 'treeID'})

In [ ]:
df_2020_2_zonal

In [ ]:
# perform a left join on the 'key' column
df_spec = pd.merge(df_2018_1_zonal[['treeID', '2018_R']], df_2018_2_zonal[['treeID', '2018_G']], on='treeID', how='left')
df_spec = pd.merge(df_spec, df_2018_3_zonal[['treeID', '2018_B']], on='treeID', how='left')
df_spec = pd.merge(df_spec, df_2018_4_zonal[['treeID', '2018_N']], on='treeID', how='left')
df_spec = pd.merge(df_spec, df_2020_1_zonal[['treeID', '2020_R']], on='treeID', how='left')
df_spec = pd.merge(df_spec, df_2020_2_zonal[['treeID', '2020_G']], on='treeID', how='left')
df_spec = pd.merge(df_spec, df_2020_3_zonal[['treeID', '2020_B']], on='treeID', how='left')
df_spec = pd.merge(df_spec, df_2020_4_zonal[['treeID', '2020_N']], on='treeID', how='left')
df_spec

In [ ]:
# create a boolean mask for NaN values
nan_mask = df_spec.isna().any(axis=1)

# filter the DataFrame to keep only rows with NaN values
nan_rows = df_spec[nan_mask]

# print the rows with NaN values
print(nan_rows)

In [ ]:
# set the threshold value
threshold = 19999

# split the dataframe based on the threshold value
df_spec_w = df_spec[df_spec['treeID'] <= threshold]  # dataframe with age <= threshold
df_spec_e = df_spec[df_spec['treeID'] > threshold]  # dataframe with age > threshold
df_spec_e

In [ ]:
dfWestMerge = pd.merge(dfWestMerge, df_spec_w, on='treeID', how='left')
dfEastMerge = pd.merge(dfEastMerge, df_spec_e, on='treeID', how='left')

In [ ]:
dfWestMerge['averageNDVI'] = ((dfWestMerge['2018_N'] - dfWestMerge['2018_R']) / (dfWestMerge['2018_N'] + dfWestMerge['2018_R'])+(dfWestMerge['2020_N'] - dfWestMerge['2020_R']) / (dfWestMerge['2020_N'] + dfWestMerge['2020_R']))/2
dfEastMerge['averageNDVI'] = ((dfEastMerge['2018_N'] - dfEastMerge['2018_R']) / (dfEastMerge['2018_N'] + dfEastMerge['2018_R'])+(dfEastMerge['2020_N'] - dfEastMerge['2020_R']) / (dfEastMerge['2020_N'] + dfEastMerge['2020_R']))/2

In [ ]:
# check column names
print(dfEastMerge.columns)
print(dfWestMerge.columns)

In [ ]:
# create a boolean mask for NaN values
nan_mask = dfEastMerge.isna().any(axis=1)

# filter the DataFrame to keep only rows with NaN values
nan_rows = dfEastMerge[nan_mask]

# print the rows with NaN values
nan_rows

In [ ]:
# create a boolean mask for NaN values
nan_mask = dfWestMerge.isna().any(axis=1)

# filter the DataFrame to keep only rows with NaN values
nan_rows = dfWestMerge[nan_mask]

# print the rows with NaN values
nan_rows


In [ ]:
print(len(dfEast))
print(len(dfEastArc))
print(len(dfEastMerge))
print(len(dfWest))
print(len(dfWestArc))
print(len(dfWestMerge))

## Adding Return and Intensity Layers

In [ ]:
folder_path = "/content/drive/MyDrive/XihanYAO_GISDataShare/Returns/W"
file_names = sorted(os.listdir(folder_path))  # sort file names alphabetically
# dfs = {}  # empty list to store dataframes

# loop through all files in folder
for filename in file_names:
    file_path = os.path.join(folder_path, filename)
    df = pd.read_csv(file_path)  # read CSV file as dataframe
    df.drop_duplicates(subset=['name'], keep='first', inplace=True)
    df = df.loc[:, ['name', 'all']]
    df['all'] = df['all'].astype(int)
    df.reset_index(drop=True, inplace=True)
    df['name'] = df['name'] + 10000
    columnname= filename[1:-8]
    df.rename(columns={'all': columnname}, inplace=True)
    dfWestMerge = pd.merge(dfWestMerge, df, how='left', left_on='treeID', right_on="name")
    dfWestMerge.drop(columns=['name'], inplace=True)
    # dfs[filename[1:-8]] = df  # add dataframe to list

# print list of dataframes
# print(dfs)
dfWestMerge['NonFirstoverFirst'] = (dfWestMerge['2re']+dfWestMerge['3re']+dfWestMerge['4re'])/dfWestMerge['1re'] 
max_val = np.nanmax(dfWestMerge['NonFirstoverFirst'])
dfWestMerge['NonFirstoverFirst'] = dfWestMerge['NonFirstoverFirst'].replace([np.inf, -np.inf], max_val)
dfWestMerge

In [ ]:
folder_path = "/content/drive/MyDrive/XihanYAO_GISDataShare/Returns/E"
file_names = sorted(os.listdir(folder_path))  # sort file names alphabetically
# dfs = {}  # empty list to store dataframes

# loop through all files in folder
for filename in file_names:
    file_path = os.path.join(folder_path, filename)
    df = pd.read_csv(file_path)  # read CSV file as dataframe
    df.drop_duplicates(subset=['name'], keep='first', inplace=True)
    df = df.loc[:, ['name', 'all']]
    df['all'] = df['all'].astype(int)
    df.reset_index(drop=True, inplace=True)
    df['name'] = df['name'] + 20000
    columnname= filename[1:-8]
    df.rename(columns={'all': columnname}, inplace=True)
    dfEastMerge = pd.merge(dfEastMerge, df, how='left', left_on='treeID', right_on="name")
    dfEastMerge.drop(columns=['name'], inplace=True)
    # dfs[filename[1:-8]] = df  # add dataframe to list

# print list of dataframes
# print(dfs)
dfEastMerge['NonFirstoverFirst'] = (dfEastMerge['2re']+dfEastMerge['3re']+dfEastMerge['4re'])/dfEastMerge['1re']
max_val = np.nanmax(dfEastMerge['NonFirstoverFirst'])
dfEastMerge['NonFirstoverFirst'] = dfEastMerge['NonFirstoverFirst'].replace([np.inf, -np.inf], max_val)
dfEastMerge

## Add Intensity

In [ ]:
dfIntensityE = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/Intensity/EastIntensityZonal.csv')
dfIntensityE = dfIntensityE.loc[:, ['TREEID', 'Intensity']]
dfIntensityE['TREEID'] = dfIntensityE['TREEID'] + 20000
dfEastMerge = pd.merge(dfEastMerge, dfIntensityE, how='left', left_on='treeID', right_on="TREEID")
dfEastMerge.drop(columns=['TREEID'], inplace=True)
dfEastMerge

In [ ]:
dfIntensityW = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/Intensity/WestIntensityZonal.csv')
dfIntensityW = dfIntensityW.loc[:, ['TREEID', 'Intensity']]
dfIntensityW['TREEID'] = dfIntensityW['TREEID'] + 10000
dfWestMerge = pd.merge(dfWestMerge, dfIntensityW, how='left', left_on='treeID', right_on="TREEID")
dfWestMerge.drop(columns=['TREEID'], inplace=True)
dfWestMerge

## Correlation

In [ ]:
#EDA
dfEastMerge.dropna(inplace=True)
# Find the maximum non-infinite value in the 'col2' column
max_val = dfEastMerge['NonFirstoverFirst'][~np.isinf(dfEastMerge['NonFirstoverFirst'])].max()

# Replace infinite values in the 'col2' column with the maximum non-infinite value
dfEastMerge['NonFirstoverFirst'] = dfEastMerge['NonFirstoverFirst'].mask(np.isinf(dfEastMerge['NonFirstoverFirst']), max_val)

# Print out the modified DataFrame
print("The modified DataFrame with infinite values replaced in 'col2' column is:")
dfEastMerge.describe()

In [ ]:
#EDA
dfWestMerge.dropna(inplace=True)
# Find the maximum non-infinite value in the 'col2' column
max_val = dfWestMerge['NonFirstoverFirst'][~np.isinf(dfWestMerge['NonFirstoverFirst'])].max()

# Replace infinite values in the 'col2' column with the maximum non-infinite value
dfWestMerge['NonFirstoverFirst'] = dfWestMerge['NonFirstoverFirst'].mask(np.isinf(dfWestMerge['NonFirstoverFirst']), max_val)

# Print out the modified DataFrame
print("The modified DataFrame with infinite values replaced in 'col2' column is:")
dfWestMerge.describe()

In [ ]:
# Check if any value in the DataFrame is too large or infinite
if dfEastMerge.isin([np.nan, np.inf, -np.inf]).any().any():
    print("There are infinite or too large values in the DataFrame.")
else:
    print("There are no infinite or too large values in the DataFrame.")

In [ ]:
corr_matrix_west = dfWestMerge.iloc[:,6:].corr()
corr_matrix_east = dfEastMerge.iloc[:,6:].corr()
# correlations between columnes on the west
fig, ax = plt.subplots(figsize=(24,21))
sns.heatmap(corr_matrix_west, annot=True, cmap='coolwarm', ax=ax)
plt.show()

In [ ]:
# correlations between columnes on the east
fig, ax = plt.subplots(figsize=(24,21))
sns.heatmap(corr_matrix_east, annot=True, cmap='coolwarm', ax=ax)
plt.show()

In [ ]:
column_name = 'p10'
max_value = max(dfWestMerge[column_name].max(), dfEastMerge[column_name].max())

# plot histograms of a column from each DataFrame on the same plot
plt.hist(dfEastMerge[column_name], alpha=0.5, label='East', bins=20, range=(0,max_value),density=True)
plt.hist(dfWestMerge[column_name], alpha=0.5, label='West', bins=20, range=(0,max_value),density=True)
plt.xlabel('p10')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Get a list of column names from the DataFrame
column_names = dfEastMerge.columns[5:]

# Set the range for the histograms
max_value = dfEastMerge.max().max()

top_percentile = 99
bottom_percentile = 1

# Loop through each column and plot its histogram
for column_name in column_names:
    # Get the bottom and top percentiles, excluding NaNs
    percentiles = np.nanpercentile(dfEastMerge[column_name], [bottom_percentile, top_percentile])
    bottom_percentile_val, top_percentile_val = percentiles[0], percentiles[1] \
        if not np.isnan(percentiles[1]) else dfEastMerge[column_name].max()

    # Repeat for the West DataFrame
    percentiles = np.nanpercentile(dfWestMerge[column_name], [bottom_percentile, top_percentile])
    bottom_percentile_val = max(bottom_percentile_val, percentiles[0]) \
        if not np.isnan(percentiles[0]) else bottom_percentile_val
    top_percentile_val = max(top_percentile_val, percentiles[1]) \
        if not np.isnan(percentiles[1]) else dfWestMerge[column_name].max()

    # plot histograms of a column from each DataFrame on the same plot
    plt.hist(dfWestMerge[column_name], alpha=0.5, label='West', bins=20, range=(bottom_percentile_val, top_percentile_val), density=True)
    plt.hist(dfEastMerge[column_name], alpha=0.5, label='East', bins=20, range=(bottom_percentile_val, top_percentile_val), density=True)
    plt.xlabel(column_name)
    plt.ylabel('Frequency')
    plt.legend(loc='upper right')
    plt.show()

# # Loop through each column and plot its histogram
# for column_name in column_names:
#     percentile99 = np.nanpercentile(dfEast[column_name], 98) \
#     if not np.isnan(np.nanpercentile(dfEast[column_name], 98)) \
#     else dfEast[column_name].max()
#     # Repeat for the West DataFrame
#     percentile99 = max(percentile99, \
#         np.nanpercentile(dfWest[column_name], 98) \
#         if not np.isnan(np.nanpercentile(dfWest[column_name], 98)) \
#         else dfWest[column_name].max())
#     # plot histograms of a column from each DataFrame on the same plot
#     plt.hist(dfEast[column_name], alpha=0.5, label='East', bins=10, range=(0,percentile99),density=True)
#     plt.hist(dfWest[column_name], alpha=0.5, label='West', bins=10, range=(0,percentile99),density=True)
#     plt.xlabel(column_name)
#     plt.ylabel('Frequency')
#     plt.legend(loc='upper right')
#     plt.show()

In [ ]:
dfEastMerge = dfEastMerge[dfEastMerge['all'] != 0]
dfWestMerge = dfWestMerge[dfWestMerge['all'] != 0]
df = pd.concat([dfWestMerge, dfEastMerge])
df

In [ ]:
df.describe()

In [ ]:
zero_rows = df[df['all'] == 0]
id_list = zero_rows['treeID'].tolist()
print(id_list)

In [ ]:
dfEastMerge.to_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/East009701_TreeMetrics_Clean.csv', index=False)

In [ ]:
dfWestMerge.to_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/West002401_TreeMetrics_Clean.csv', index=False)

In [ ]:
df.to_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/All_TreeMetrics_Clean.csv', index=False)

In [ ]:
corr_matrix = df.iloc[:,6:].corr()
# correlations between columnes on the west
fig, ax = plt.subplots(figsize=(24,21))
sns.heatmap(corr_matrix_west, annot=True, cmap='coolwarm', ax=ax)
plt.show()

In [ ]:
# # Replace NaN values with column median
# imputer = SimpleImputer(strategy='median')
# columns = df.iloc[:, 5:].values
# columns = imputer.fit_transform(columns)

# # Perform PCA
# pca = PCA()
# pca.fit(columns)

# # Determine number of components with eigenvalues > 1
# len([eigval for eigval in pca.explained_variance_ if eigval > 1])